In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(os.path.abspath("../src"))
from wordings import all_clauses
from rag import answer_sql, load_sql, search_sql

In [0]:
dbutils.widgets.text("catalog", "reinsurance")
catalog = dbutils.widgets.get("catalog")
kb = f"{catalog}.knowledge"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {kb}")
spark.createDataFrame(all_clauses()).createOrReplaceTempView("clauses")
spark.sql(load_sql(kb, "clauses"))
display(spark.sql(f"SELECT count(*) AS clauses, min(size(embedding)) AS dimensions FROM {kb}.treaty_clauses"))

In [0]:
EVAL = [
    ("¿Qué porcentaje cede Aseguradora Sierra en autos después del endoso de 2024?", "T-CED01-MOTOR-2024", 7),
    ("¿Cuál es la retención del contrato de daños de Seguros Costa Azul de 2025?", "T-CED02-PROPERTY-2025", 2),
    ("¿Qué exclusiones tiene el contrato de transporte marítimo de Meridiano de 2024?", "T-CED03-MARINE-2024", 4),
    ("¿En cuántos días hay que avisar un siniestro del contrato de daños de Aseguradora Sierra 2025?", "T-CED01-PROPERTY-2025", 5),
    ("¿Cómo se paga la prima del contrato de daños de Meridiano de 2024?", "T-CED03-PROPERTY-2024", 3),
    ("¿Qué porcentaje cede Seguros Costa Azul en su contrato de autos de 2025?", "T-CED02-MOTOR-2025", 2),
]
TOP_K = 5

rows = []
for question, treaty, clause in EVAL:
    hits = spark.sql(search_sql(kb, TOP_K), args={"question": question}).collect()
    found = any(h.treaty_number == treaty and h.clause_no == clause for h in hits)
    rows.append((f"{treaty} #{clause}", found, f"{hits[0].treaty_number} #{hits[0].clause_no}", question))
display(spark.createDataFrame(rows, "expected string, found_in_top_k boolean, top_hit string, question string"))

In [0]:
r = spark.sql(answer_sql(kb, TOP_K), args={"question": EVAL[1][0]}).first()
print(r.answer)
print(r.sources)

In [0]:
r = spark.sql(answer_sql(kb, TOP_K), args={"question": "¿Cuál es el deducible de gastos médicos mayores?"}).first()
print(r.answer)